<a href="https://colab.research.google.com/github/anjicx/CNHypergraph/blob/main/GraphMake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hypergraph is a type of graph where one edge (hyperedge) can connect multiple nodes. They are used because simplified models (like traditional graphs) often fail to capture the reality of complex systems.
Nodes are diseases and hyperedges are patients visits.

In [2]:
#connecting to sampled data
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Diagnosis is a dictionary containing all diagnosis.Stays have foreign key of primary to dictionary; primary keys of secondary_diagnosis are diagnosis and stays.

In [3]:
import pandas as pd
#loading diagnosis

path = "/content/drive/MyDrive/PatientData/table_diagnoses.csv"
diagnosis = pd.read_csv(path, sep=";")
print(diagnosis.head())
#print(diagnosis["diagnose_id"].isna().sum())#all Primary keys have value
#print(diagnosis["diagnose_id"].duplicated().sum())#no duplicates


#loading stays and stays_secondaries
stays = pd.read_csv("/content/drive/MyDrive/PatientData/final_one_percent_stays.csv", sep=";")
stays_secondaries = pd.read_csv("/content/drive/MyDrive/PatientData/final_one_percent_stays_secondaries.csv", sep=";")

   diagnose_id                          descr icd_code
0        14648               Pfannenlockerung     1010
1        14649            OsteolyseAcetabulum     1011
2        14650  GroßerKnochendefektAcetabulum     1012
3        14651   ChondropathiebeiKopfprothese     1013
4        14652       FehlpositionierungPfanne     1014


Creating disease nodes from all possible diagnosis. In nodes we have node_id as index and icd_code regarding disease.

In [4]:
nodes=diagnosis.copy()
nodes["node_id"] = range(len(nodes))#LATER FOR NODE_INDEX
nodes = nodes[["node_id", "diagnose_id", "descr", "icd_code"]]#NODE INDEX AT THE BEGINNING
nodes.head()

,node_id,diagnose_id,descr,icd_code
0,0,14648,Pfannenlockerung,1010
1,1,14649,OsteolyseAcetabulum,1011
2,2,14650,GroßerKnochendefektAcetabulum,1012
3,3,14651,ChondropathiebeiKopfprothese,1013
4,4,14652,FehlpositionierungPfanne,1014


Creating a table with visits+pri/secon diagnosis

In [5]:
#creating a tbl of primary diagnosis
primary = stays[["stay_id", "pri_diag_id"]].copy()#stays has for key to diagnosis
primary = primary.rename(columns={"pri_diag_id": "diagnose_id"})#rename foreign key diagnose_id
primary["role"]="primary"

#creating a tbl of secondary diagnosis
secondary = stays_secondaries[["stay_id", "sec_diag_id"]].copy()
secondary = secondary.rename(columns={"sec_diag_id": "diagnose_id"})
secondary["role"]="secondary"

#union of all primary and secondary diseases - just one after another
stay_diagnoses = pd.concat([primary, secondary], ignore_index=True)

#inner join
stay_diagnoses = pd.merge(stay_diagnoses,nodes,on="diagnose_id",how="inner")

#stay_diagnoses["node_id"] = stay_diagnoses["node_id" ].astype(int)# IF WE DO LEFT JOIN NAN VALUE+INTEGER -> FLOAT



Creating 1 hyperedge table from the table of visits+diagnosis where it's not 1 it's assumed as 0

In [6]:
# 1. all unique stays
unique_stays = stay_diagnoses["stay_id"].drop_duplicates().reset_index(drop=True) # all unique stays because 1 stay is multiple time so to map 1 stay hyperedge with diagnosis--62,871,22.. visits

# 2. for each visit create visit_id:hyperedge_id ; key=visit value=hyperedge index
stay_to_hyperedge = {}
for i in range(len(unique_stays)):
  stay_to_hyperedge[unique_stays[i]]=i#hyperedge of visit is i

# 3. map stay_id to visit adding it to the table stay_diagnosis
stay_diagnoses["hyperedge_index"]=stay_diagnoses["stay_id"].map(stay_to_hyperedge)

# 4. create only node indx:hyperedge indx table
hyperedge_table=stay_diagnoses[["node_id","hyperedge_index"]].copy()



In [8]:
hyperedge_table.head(4)

,node_id,hyperedge_index
0,8352,0
1,1257,1
2,428,2
3,2054,3


Features related to nodes/diagnoses and hyperedges/visits